# Document features / 文档特性

This example combines local font/color resources, rich text, graphics, raw reaction references, and irregular typed fields.

本示例组合局部字体/颜色资源、富文本、图形、原始反应引用及不规则类型字段。

In [1]:
from cdxml_om import (
    ArrowheadSide,
    ArrowheadType,
    BoundingBox,
    CDXMLDocument,
    ElementList,
    GenericList,
    GraphicType,
    Point2D,
    Point3D,
    SpectrumClass,
    SpectrumXType,
    SpectrumYType,
    TagType,
)

document = CDXMLDocument.from_string("<CDXML/>")
font_table = document.create_font_table()
font = font_table.fonts.create(name="Example Sans", charset="iso-8859-1")
color_table = document.create_color_table()
color_table.colors.create(r=0.15, g=0.35, b=0.75)
page = document.pages.create()
group = page.groups.create()
text = group.texts.create(position=(12.0, 18.0))
run = text.runs.create(content="Typed text", font_id=font.id, size=12.5, color=0)
second_run = text.runs.create(content=" emphasized", font_id=font.id, size=9.0, face=1, color=0)
run.size = 13.0
fragment = group.fragments.create()
carbon = fragment.nodes.create(element=6, position=(0.0, 0.0))
oxygen = fragment.nodes.create(element=8, position=(12.0, 0.0))
fragment.bonds.create(begin=carbon, end=oxygen, order=1)
graphic = group.graphics.create(
    graphic_type=GraphicType.RECTANGLE, bounding_box=(2.0, 3.0, 18.0, 14.0)
)
arrow = group.arrows.create(
    head_3d=(9.0, 4.0, 0.0),
    tail_3d=(1.0, 4.0, 0.0),
    arrowhead_type=ArrowheadType.SOLID,
    arrowhead_head=ArrowheadSide.FULL,
)
assert run.font_id == font.id and run.color == 0
assert text.plain_text == "Typed text emphasized"
assert second_run.content == " emphasized" and second_run.size == 9.0 and second_run.face == 1
print(f"Text has {len(text.runs)} styled runs; font ID {run.font_id}, raw color index {run.color}")

Text has 2 styled runs; font ID 1, raw color index 0


## Irregular fields and references / 不规则字段与引用

Curve coordinate lists and object tags use explicit codecs. Reaction-step lists retain generic object references; they are not chemical role validation.

曲线坐标列表和对象标签使用显式 codec。反应步骤列表保留通用对象引用，不负责判断化学角色是否合理。

In [2]:
curve = fragment.curves.create(
    curve_points=(Point2D(0.0, 1.0), Point2D(2.0, 3.0)),
    curve_points3_d=(Point3D(0.0, 1.0, 2.0), Point3D(3.0, 4.0, 5.0)),
)
tag = group.objecttags.create(name="measurement", tag_type=TagType.DOUBLE, value=2.5)
tag.value = 3.75
run.content = "Updated"
element_list = ElementList(elements=(6, 8), negated=True)
generic_list = GenericList(values=("alpha", "beta"))
carbon.element_list = element_list
oxygen.generic_list = generic_list
scheme = group.schemes.create()
step = scheme.steps.create()
step.reactants = (fragment, group)
step.products = (fragment,)
step.arrows = (arrow, graphic)
assert tuple((point.x, point.y) for point in curve.curve_points or ()) == ((0.0, 1.0), (2.0, 3.0))
assert curve.curve_points3_d is not None and len(curve.curve_points3_d) == 2
assert tag.value == 3.75
assert element_list.negated and element_list.elements == (6, 8)
assert generic_list.values == ("alpha", "beta")
assert carbon.element_list == element_list and oxygen.generic_list == generic_list
assert text.plain_text == "Updated emphasized"
assert step.reactants == (fragment, group)
assert step.raw_reference_ids("reactants") is not None
print(f"Curve contains {len(curve.curve_points or ())} 2D points; tag value is {tag.value}")

Curve contains 2 2D points; tag value is 3.75


## Spectrum storage and round trip / 光谱存储与往返

Spectrum `.data` is direct lexical PCDATA, not a decoded numeric sample array. Font IDs and text color indices are local/raw indices, not linked resource wrappers.

光谱 `.data` 是直接的词法 PCDATA，并非已解码的数值采样数组。字体 ID 与文本颜色索引是局部/原始索引，不是资源 wrapper。

In [3]:
spectrum = group.spectrums.create(
    class_name=SpectrumClass.UNKNOWN,
    x_type=SpectrumXType.OTHER,
    y_type=SpectrumYType.OTHER,
    x_low=0.0,
    y_low=0.0,
    y_scale=1.0,
    data="400 0.2 500 0.5",
)
spectrum.annotations.create(content="Illustrative lexical payload")
assert spectrum.data == "400 0.2 500 0.5"
assert document.validate().is_valid, document.validate().errors
reloaded = CDXMLDocument.from_string(document.to_string())
reloaded_group = reloaded.pages[0].groups[0]
reloaded_step = reloaded_group.schemes[0].steps[0]
assert reloaded_step.arrows[0].xml_tag == "arrow"
reloaded_fragment = reloaded_group.fragments[0]
assert reloaded_step.reactants == (reloaded_fragment, reloaded_group)
assert reloaded_step.products == (reloaded_fragment,)
assert reloaded_step.arrows == (reloaded_group.arrows[0], reloaded_group.graphics[0])
reloaded_text = reloaded_group.texts[0]
assert [run.content for run in reloaded_text.runs] == ["Updated", " emphasized"]
assert [run.size for run in reloaded_text.runs] == [13.0, 9.0]
assert reloaded_text.runs[1].face == 1 and reloaded_text.runs[1].font_id == font.id
assert reloaded.font_table.fonts[0].name == "Example Sans"
assert reloaded.font_table.fonts[0].charset == "iso-8859-1"
assert reloaded.color_table.colors[0].r == 0.15 and reloaded.color_table.colors[0].g == 0.35
assert reloaded_group.graphics[0].bounding_box == BoundingBox(2.0, 3.0, 18.0, 14.0)
assert reloaded_group.arrows[0].head_3d == Point3D(9.0, 4.0, 0.0)
assert reloaded_group.arrows[0].tail_3d == Point3D(1.0, 4.0, 0.0)
assert reloaded_fragment.curves[0].curve_points == curve.curve_points
assert reloaded_fragment.curves[0].curve_points3_d == curve.curve_points3_d
assert reloaded_group.spectrums[0].data == spectrum.data
assert reloaded_group.spectrums[0].annotations[0].content == "Illustrative lexical payload"
assert reloaded_group.objecttags[0].value == 3.75
assert reloaded_group.fragments[0].nodes[0].element_list == element_list
assert reloaded_group.fragments[0].nodes[1].generic_list == generic_list
assert reloaded.validate().is_valid, reloaded.validate().errors
print(
    f"Reloaded {len(reloaded.font_table.fonts)} fonts and {len(reloaded.color_table.colors)} colors"
)
print("Rich text, geometry, references, irregular fields, and Spectrum round-trip.")
print("CDXML-OM notebook: 02_document_features OK")

Reloaded 1 fonts and 1 colors
Rich text, geometry, references, irregular fields, and Spectrum round-trip.
CDXML-OM notebook: 02_document_features OK
